In [1]:
# Regressor와 Classifier의 차이
# Regressor = 연속된 숫자 중 숫자 하나만 출력-> mae, mse, r2 스코어(실제 값을 얼마나 잘 설명했냐) 출력 가능
# Classifier =  연속된 구간 중 하나의 구간만 출력-> f-1 스코어(실제 구간에서 얼마나 잘 분류했냐) 출력 가능
# 이용객수는 다양한 구간으로 나누어 실험해봤는데 "0~700/700~3000/3000 이상"이 가장 잘 설명함
#
# 특이사항 1:   중간 구간(700~3천) 구간이 f1스코어가 다른 구간보다 낮게 출력됨.
#              표본 개수 문제는 아니고 아마 피처 작용하는 것 중 수치로는 알 수 없는 무언가가 있는 것으로 예상(습도, 휴관, 샌드위치 등등등)
#
# 특이사항 2:   구간별로 큰 차이가 없어서 class_weight = "balanced"는 굳이 안해도 될 듯 함. 
#               캣부스트의 경우 오히려 전체 f-1 스코어는 오르지만 중간 구간은 조금 떨어지는 상황 발생
    
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report

# 아래는 쓰지 않을 모델들
#from sklearn.preprocessing import StandardScaler
#from sklearn.linear_model import LogisticRegression
#from sklearn.neighbors import KNeighborsClassifier
#from sklearn.compose import ColumnTransformer
#from sklearn.preprocessing import OneHotEncoder, StandardScaler
#from sklearn.pipeline import Pipeline
df = pd.read_csv("Oworld.csv", encoding="utf-8-sig")


X = df[["holiday","mon","tue","wed","thur","fri","sat","sun","temperature","rain","humidity"]]

#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~1500명 :혼잡
#1500 이상 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 랜덤포레스트, 캣부스트
models = {
    #선형회귀, KNN 주석 풀면 적용 가능
    #"Logistic Regression": LogisticRegression(max_iter=1000),
    #"KNN": KNeighborsClassifier(n_neighbors=5),

    "Random Forest": RandomForestClassifier(
        #class_weight="balanced",
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0

    )

}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)

    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.67      0.86      0.75       102
   700~3000명       0.64      0.52      0.57        85
    3001명 이상       0.91      0.68      0.78        57

    accuracy                           0.70       244
   macro avg       0.74      0.69      0.70       244
weighted avg       0.71      0.70      0.70       244

              precision    recall  f1-score   support

      0~700명       0.73      0.87      0.79       102
   700~3000명       0.64      0.64      0.64        85
    3001명 이상       0.97      0.63      0.77        57

    accuracy                           0.73       244
   macro avg       0.78      0.71      0.73       244
weighted avg       0.75      0.73      0.73       244

           Model   f1
0       CatBoost 0.73
1  Random Forest 0.70
